# 03 · Train the conditional diffusion model

The diffusion counterpart of pix2pix (Palette-style; Saharia et al., 2022). It sees the same
tiles, the same 9-channel condition and the same train/validation/test split, so the two
models can be compared fairly in `04_evaluate.ipynb`. Runtime: **GPU (T4)**, about 3–4 hours
for the configured epochs. Checkpoints go to Drive every epoch: if Colab disconnects, re-run
this notebook and it resumes.

In [ ]:
from google.colab import drive

REPO_URL = "https://github.com/muhammedshahidsds25-collab/kerala-hazard-vision-ai.git"
REPO_DIR = "/content/kerala-hazard-vision-ai"

drive.mount("/content/drive")
!git clone -q {REPO_URL} {REPO_DIR} || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# Regular (not editable) install: an editable src-layout install is not importable
# until the runtime restarts.
!pip install -q .

In [ ]:
import dataclasses
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from kerala_hazard_vision_ai.config import load_config
from kerala_hazard_vision_ai.train_diffusion import fit_diffusion

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
TILE_DIR = DRIVE_ROOT / "tiles"
CKPT_DIR = DRIVE_ROOT / "checkpoints_diffusion"
FIG_DIR = DRIVE_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only: use a GPU runtime")

## Training (resumes automatically)

In [ ]:
run_cfg = dataclasses.replace(
    cfg, diffusion=dataclasses.replace(cfg.diffusion, checkpoint_dir=str(CKPT_DIR))
)
best_weights = fit_diffusion(run_cfg, TILE_DIR, resume=True)
best_weights

## Loss curves
The denoising loss falls quickly and then flattens; validation uses fixed noise, so its curve
is comparable across epochs. The EMA weights with the lowest validation loss are kept.

In [ ]:
history = pd.read_csv(CKPT_DIR / "history.csv")
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], logy=True, figsize=(6, 4),
                  title="Diffusion denoising loss (MSE on noise)")
ax.figure.savefig(FIG_DIR / "losses_diffusion.png", dpi=150)
plt.show()
history.tail()